In [ ]:
import pandas as pd
s = pd.read_csv("sales_transactions.csv", parse_dates=["date"])
p = pd.read_csv("promotion_calendar.csv", parse_dates=["start_date", "end_date"])
s.info()
p.info()
s.describe()

In [ ]:
s["disc"] = s.discount_rate.where(s.discount_rate <= 1, s.discount_rate / 100)
s["month"] = s.date.dt.month
s["net"] = s.quantity * s.unit_price * (1 - s.disc)
s["profit"] = s.net - s.quantity * s.unit_cost
s["margin"] = s.profit / s.net
s["diff"] = s.net_revenue - s.net
s[s["diff"].abs() > 0.01]

In [ ]:
display(s.pivot_table("disc", "category", "month").round(2))
g = s.groupby(["category", "month"]).disc
q1, q3, n = g.transform("quantile", 0.25), g.transform("quantile", 0.75), g.transform("size")
s["iqr"] = (n >= 20) & (s.disc > q3 + 1.5 * (q3 - q1))
print((g.size() < 20).sum(), (n < 20).sum())
s[s.iqr]

In [ ]:
v = s[s.promotion_id.notna()].merge(p, how="left")
v["id_ok"] = v.max_discount.notna()
v["cat_ok"] = v.category == v.applicable_category
v["date_ok"] = v.date.between(v.start_date, v.end_date)
v["max_ok"] = v.disc <= v.max_discount
v["plan_ok"] = (v.disc - v.planned_discount).abs().round(4) <= 0.05
c = ["id_ok", "cat_ok", "date_ok", "max_ok", "plan_ok"]
bad = v[v[c].all(axis=1) == False]
bad[["promotion_id", "category", "disc", "iqr"] + c]

In [ ]:
ok = v[v.id_ok & v.cat_ok & v.date_ok].transaction_id
s["valid"] = s.transaction_id.isin(ok)
s["over30"] = (s.disc > 0.3) & (s.valid == False)
s["low"] = s.margin < 0.15
print(s.low.sum(), s[s.low].valid.sum())
s[s.over30]

In [ ]:
def impact(by):
    t = s.groupby(by).agg(n=("net", "size"), sales=("net", "sum"), profit=("profit", "sum"))
    t["margin"] = t.profit / t.sales
    return t.round(2)
display(impact("valid"))
display(impact(["category", "valid"]))
impact(s.promotion_id.fillna("None"))

In [ ]:
k = ["net_revenue", "net", "diff"]
r = s.groupby("category")[k].sum()
r.loc["Sum"] = r.sum()
r.loc["Total"] = s[k].sum()
r.round(2)

In [ ]:
s["promo_issue"] = s.transaction_id.isin(bad.transaction_id)
s["data_issue"] = (s["diff"].abs() > 0.01) | (s.discount_rate > 1)
f = ["iqr", "data_issue", "promo_issue", "over30", "low"]
s[s[f].any(axis=1)].to_csv("flagged_transactions.csv", index=False)
res = s[f].sum()
res["flagged"] = s[f].any(axis=1).sum()
res.to_csv("summary_results.csv")
res

## Summary
- Data: 8 discounts saved as percentages, fixed in disc. 6 sales without the discount (+$58.35), kept as recorded.
- IQR: 15 outliers. 11 small Toys groups (135 sales) not checked.
- Promotions: 17 of 188 fail the calendar checks.
- Rules: 10 over 30% without a valid promotion. 72 under the 15% margin.
- Impact: valid promotions keep a 17% margin vs 33%. PRM-07 kept 3%.
- Final notes: synthetic files generated with AI before recording. Originals unchanged.